In [ ]:
import pandas as pd

# 1. Load the WHO dataset
who_df = pd.read_csv("/content/WHO-COVID-19-global-monthly-death-by-age-data.csv")

# 2. Fill empty/missing death rows with 0
who_df['Deaths'] = who_df['Deaths'].fillna(0)

# 3. Explicitly convert columns to ensure proper numeric operations
who_df['Year'] = who_df['Year'].astype(int)
who_df['Month'] = who_df['Month'].astype(int)
who_df['Deaths'] = who_df['Deaths'].astype(int)

print("WHO Data Cleaned Successfully!")

WHO Data Cleaned Successfully!


In [ ]:
# 1. Load the vaccination dataset
vaccine_df = pd.read_csv("/content/vaccinations.csv")

# 2. Convert the 'date' column into a proper datetime object
vaccine_df['date'] = pd.to_datetime(vaccine_df['date'])

# 3. Extract Year and Month columns so they match the WHO format
vaccine_df['Year'] = vaccine_df['date'].dt.year
vaccine_df['Month'] = vaccine_df['date'].dt.month

# 4. Filter out global/regional aggregate rows (non-country codes starting with OWID)
vaccine_df = vaccine_df[~vaccine_df['iso_code'].str.startswith('OWID', na=False)]

# 5. Group by Country and Month, capturing the maximum cumulative vaccination coverage reached
clean_vaccine_df = vaccine_df.groupby(['iso_code', 'Year', 'Month']).agg({
    'people_fully_vaccinated_per_hundred': 'max',
    'total_boosters_per_hundred': 'max'
}).reset_index()

# 6. Fill any remaining NaNs with 0 (for periods before vaccines were introduced)
clean_vaccine_df['people_fully_vaccinated_per_hundred'] = clean_vaccine_df['people_fully_vaccinated_per_hundred'].fillna(0)
clean_vaccine_df['total_boosters_per_hundred'] = clean_vaccine_df['total_boosters_per_hundred'].fillna(0)

print("Vaccination Data Aggregated and Cleaned!")

Vaccination Data Aggregated and Cleaned!


In [ ]:
# Merge the clean WHO data with the aggregated monthly vaccine data
# We map 'Country_code' from WHO to 'iso_code' from the vaccine data
master_df = pd.merge(
    who_df,
    clean_vaccine_df,
    left_on=['Country_code', 'Year', 'Month'],
    right_on=['iso_code', 'Year', 'Month'],
    how='left' # 'left' keeps all records from our primary WHO death dataset
)

# If a country had no vaccine records for early 2020 months, fill those missing values with 0
master_df['people_fully_vaccinated_per_hundred'] = master_df['people_fully_vaccinated_per_hundred'].fillna(0)
master_df['total_boosters_per_hundred'] = master_df['total_boosters_per_hundred'].fillna(0)

# Drop the duplicate iso_code column from the merge
master_df = master_df.drop(columns=['iso_code'])

# Save this out as your dashboard's engine
master_df.to_csv("cleaned_covid_dashboard_data.csv", index=False)
print("Master Dashboard Dataset Ready for Streamlit!")

Master Dashboard Dataset Ready for Streamlit!
